<a href="https://colab.research.google.com/github/Irtisam99/Deep_Learning/blob/main/Multilayer_Perceptron_(MNIST)_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

In [2]:
""" Deep Neural Network / Artificial Neural Network
1. Feed Forwad Neural Network  (Fully Connected normally)
    Example: Input Layer> Hidden Layer > Output Layer
2. Convulational Neural Network
    Example: Input layer (Image) > CNN layer (s) > Fully Connected Layer(s) > output layer
            Image (300 x 300)
            Raw features: 90000 pixels. Each pixel is a feature.
            CNN layers extracts feature maps from the image
3. Recurrent Neural Network: Sequential Data uses forward feed connection,The output of a neuron is looped back into itself as part of the input for the next step in the sequence.
      Hi, His name is Irtisam. He studies computer science. He ___
"""
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


cuda


In [3]:
""" Image -> class

Image: A 2D matrix of pixel values.
Color Channels:
  - Grayscale image has 1 channel per pixel. For a pixel color between (0-255)
    [ 125 124  15 155
      145 145  245 87]
  - RGB image has 3 channels per pixel. For a pixel (0-255, 0-255, 0-255)
    [ (125, 145, 147) ....
      ...................]
"""
# Preprocessing pipeline
from torchvision import transforms,datasets

"""
Localized normalization:
   image is normalized with its own mean and std
Globalized Normalization:
   image is normalized with all the images mean and std
"""

transform=transforms.Compose([
    transforms.ToTensor(),  # Scales pixels values from 0-255 to 0-1
    transforms.Normalize(mean=(0.1307,),std=(0.3081))  # Globalized values for MNIST, shifts the data so that mean becomes 0 and std becomes 1 overall
])


In [4]:
# Download dataset
train_dataset=datasets.MNIST(
    root='data',
    train=True,
    download=True,
    transform=transform

)
test_dataset=datasets.MNIST(
    root='data',
    train=False,
    download=True,
    transform=transform
)

In [5]:
"""
MNIST: Hand written digit recognition dataset
Trainset contains 60,000 images
Testset contains 10,000 images
Each image is 28x28 in shape
So there is 784 pixels for each image

We are given 785 columns for each image
X: column 1-784 (pixels)
y: column 785 (digits)

Model training steps in each epoch:
   1. Forward propagation / Predicts the outcome given the input values / logits
   2. Calculate the loss
   3. Calculate Gradient w.r.t weights using backpropagation
   4. Update weights

layer 1: w1
layer 2: y = f(w1)
layer 3: z = f(w2)
loss = loss_fn(z, actual_z)

We need to know how much each weight (w1, w2) contributed to the error.

dJ
___
dw2

dJ      dJ       dw2
___ = ______ x _____
dw1     dw2      dw1

Batch
=======
We have 60000 training images
We want 10 epochs to run for training

Gradient Descent (GD) is an optimization algorithm that minimizes a model's cost function by updating parameters iteratively.
Batch GD uses the entire dataset, offering precise convergence but slow speed.
Stochastic GD (SGD) uses one example per update, allowing fast, frequent updates that are noisy.
Mini-Batch GD balances these, using small data subsets for faster, stabler convergence, and is the standard approach in deep learning. Batch size 32, 64, 128, 16, 8, 4


for epoch in epochs:
    train_batches = randomly distribute the training samples into batches
    test_batches = randomly distribute the test samples into batches
    for train_batch in train_batches:
        1. Predict the outcome given train_batch inputs
        2. Calculate loss
        3. Calculate gradients w.r.t weights
        4. Update weights
        5. Validate the performance on the test_batches

A batch size is a random sample from the training set.
batch size 32, 64, 128
"""
from torch.utils.data import DataLoader

train_loader=DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader=DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [6]:
class Perceptron(nn.Module):
  def __init__(self,input_size):
    super(Perceptron,self).__init__()
    self.w=nn.Parameter(torch.randn(input_size))
    self.b=nn.Parameter(torch.randn(1))

  def forward(self,x):
    x=x @ self.w+self.b
    return x

In [ ]:
input_size=28*28
sample_input=torch.randn(input_size)
sample_input

In [8]:
model=Perceptron(input_size)
output=model(sample_input)
print(output)

tensor([8.0518], grad_fn=<AddBackward0>)


In [9]:
class ReLU(nn.Module):
  def __init__(self):
    super(ReLU,self).__init__()

  def forward(self,x):
    return torch.maximum(torch.tensor(0.0),x)



In [10]:
relu=ReLU()
output=relu(output)
print(output)

tensor([8.0518], grad_fn=<MaximumBackward0>)


In [11]:
# Creating a single linear layer of multiple neurons

class Linear(nn.Module):
  def __init__(self,input_size,output_size):
    super(Linear,self).__init__()
    self.perceptrons=nn.ModuleList(
        [Perceptron(input_size) for _ in range(output_size)]
    )
  def forward(self,x):
    output=[perceptron(x) for perceptron in self.perceptrons]
    outputs=torch.stack(output,dim=1)
    return outputs

In [18]:
class DigitClassifier(nn.Module):
  def __init__(self,input_size=784,output_size=10):
    super(DigitClassifier,self).__init__()
    self.fc1=Linear(input_size,512)
    self.fc2=Linear(512,256)
    self.fc3=Linear(256,128)
    self.fc4=Linear(128,output_size)
    self.relu=nn.ReLU()
  def forward(self,x):
    #print(x.shape) # B,1,28,28
    x=x.view(-1,input_size)  #(B,784)
    #print(x.shape)
    x=self.fc1(x)
    x=self.relu(x)
    x=self.fc2(x)
    x=self.relu(x)
    x=self.fc3(x)
    x=self.relu(x)
    x=self.fc4(x)
    return x

In [19]:
""" Use of Device
Data is stored in (RAM/GPU) when program is running
model is also loaded(RAM/GPU)
"""
model =DigitClassifier(input_size=28*28,output_size=10).to(device)
sample_input=sample_input.to(device)

output=model(sample_input)
print(output.shape)
print(output)

torch.Size([1, 10])
tensor([[-45050.9141, -79643.0000,  22347.1367, -38891.3906, -39020.4531,
         -38956.2773,  56499.0547, -29511.9297,  79106.0391, -50047.5352]],
       device='cuda:0', grad_fn=<StackBackward0>)


In [20]:
total_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters:{total_params:,}")

Total Trainable Parameters:567,434


In [21]:
((28*28)+1)*512+(512+1)*256+(256+1)*128+(128+1)*10

567434

In [29]:
param_size=next(model.parameters()).element_size()
model_size_bytes=total_params*param_size
model_size_mb=model_size_bytes/(1024*1024)

In [30]:
print(f"Total Trainable params:  {total_params:,}")
print(f"Parameter Size(float_32):{param_size} bytes")
print(f"Estimated Model Size:    {model_size_mb:.2f} MB")
print(f"Estimated Size (float16):{model_size_mb/2:.2f} MB")

Total Trainable params:  567,434
Parameter Size(float_32):4 bytes
Estimated Model Size:    2.16 MB
Estimated Size (float16):1.08 MB


In [31]:
max_val,predicted_id=torch.max(output,1)
print(max_val)
print(predicted_id)

tensor([79106.0391], device='cuda:0', grad_fn=<MaxBackward0>)
tensor([8], device='cuda:0')


In [32]:
import torch.optim as optim

optimizer=optim.Adam(model.parameters(),lr=0.001)
criterion=nn.CrossEntropyLoss()